In [1]:
## COUNTY TAXES TABLES
city_to_county = {
    # Utah County
    "provo": "Utah County", "orem": "Utah County", "lehi": "Utah County",
    "spanish fork": "Utah County", "springville": "Utah County",
    "american fork": "Utah County", "saratoga springs": "Utah County",
    "pleasant grove": "Utah County", "payson": "Utah County",
    "mapleton": "Utah County", "highland": "Utah County",
    "eagle mountain": "Utah County", "cedar hills": "Utah County",
    "lindon": "Utah County", "salem": "Utah County", "santaquin": "Utah County",
    "vineyard": "Utah County", "alpine": "Utah County",
    "elk ridge": "Utah County", "woodland hills": "Utah County",
    "bluffdale": "Utah County", "draper": "Utah County",

    # Salt Lake County
    "salt lake city": "Salt Lake County", "millcreek": "Salt Lake County",
    "sandy": "Salt Lake County", "west jordan": "Salt Lake County",
    "south jordan": "Salt Lake County", "taylorsville": "Salt Lake County",
    "murray": "Salt Lake County", "holladay": "Salt Lake County",
    "midvale": "Salt Lake County", "cottonwood heights": "Salt Lake County",
    "riverton": "Salt Lake County", "south salt lake": "Salt Lake County",
    "west valley city": "Salt Lake County", "magna": "Salt Lake County",
    "kearns": "Salt Lake County", "herriman": "Salt Lake County",
    "brighton": "Salt Lake County", "emigration canyon": "Salt Lake County",
    "copperton": "Salt Lake County", "alta": "Salt Lake County"
}

county_tax_rates = {
    "Beaver County": 0.0042, "Box Elder County": 0.0054,"Cache County": 0.0051, "Carbon County": 0.0072,
    "Daggett County": 0.0045,"Davis County": 0.0057,"Duchesne County": 0.0066,"Emery County": 0.0061,
    "Garfield County": 0.0040,"Grand County": 0.0040,"Iron County": 0.0046,"Juab County": 0.0049,"Kane County": 0.0045,
    "Millard County": 0.0052,"Morgan County": 0.0053,"Piute County": 0.0049,"Rich County": 0.0034,
    "San Juan County": 0.0084,"Sanpete County": 0.0054,"Sevier County": 0.0059,"Summit County": 0.0034,
    "Tooele County": 0.0060,"Uintah County": 0.0055,
    "Wasatch County": 0.0048,
    "Washington County": 0.0046,
    "Wayne County": 0.0038, "Weber County": 0.0063,
    # SLC
    "Salt Lake County": 0.0059,
    # UTAH
    "Utah County": 0.0045
}

In [2]:
## CODE TO SCRAPE PAGE BY PAGE
import pyzill
import json
import csv
from datetime import datetime

##CODE TO PULL UNIQUE LISTINGS FROM CITIES + CITY TAG + GRM + SqFtMultiplier
import csv
import openpyxl
from openpyxl import Workbook
from geopy import Point
from geopy.distance import geodesic
from datetime import datetime


#USER INPUTS
Per_bed_rent = 600 #ASSUMPTION ON $ / bedroom for rent multiplier
Down_pmt_per = .25
interest_rate = .067
insurance_rate = .0043


# Define Monthly Payment Formula
def PMT(loan_amount, interest_rate):
    monthly_rate = interest_rate / 12
    nper = 30 * 12
    if monthly_rate == 0:  # handle zero interest
        return loan_amount / nper
    return loan_amount * (monthly_rate * (1 + monthly_rate) ** nper) / ((1 + monthly_rate) ** nper - 1)

def PITI(loan_amount, county_tax_rate, insurance_rate, home_price):
    taxes = county_tax_rate * home_price /12 #monthly amount
    insurance = insurance_rate * home_price /12 #monthly amount
    monthly_rate = interest_rate / 12
    nper = 30 * 12
    if monthly_rate == 0:
        return taxes + insurance + loan_amount / nper
    monthly_pmt = loan_amount * (monthly_rate * (1 + monthly_rate) ** nper) / ((1 + monthly_rate) ** nper - 1)
    return taxes + insurance + monthly_pmt


# Get the current timestamp for unique filenames
current_time = datetime.now().strftime("%Y_%m%d_%H%M")

# Define the parameters and list of cities with their center coordinates and target area
target_listings = 1000
zoom_value = 1
cities = [
    {"name": "Millcreek", "lat": 40.682460, "long": -111.846810, "target_area": 9},
    {"name": "Avenues", "lat": 40.780370, "long": -111.875450, "target_area": 4},
    {"name": "Salt Lake City", "lat": 40.7608, "long": -111.8910, "target_area": 100},
    {"name": "Provo", "lat": 40.2338, "long": -111.6585, "target_area": 40},
    # {"name": "Seattle", "lat": 47.6062, "long": -122.3321, "target_area": 25}
]

# Generate a simplified filename with a timestamp and city count
city_count = len(cities)
file_name = f"{current_time}_{city_count}_cities_MF_data.xlsx"

# Initialize tracking variables for unique IDs and combined results
unique_ids = set()
combined_results = []

# Define fields for Excel output
fields = [
    'zpid', 'rawHomeStatusCd', 'homeStatusForHDP', 'homeStatus',
    'homeType', 'address', 'addressZipcode', 'addressCity', 'addressState',
    'latitude', 'longitude', 'detailUrl', 'unformattedPrice', 'beds', 'baths',
    'area', 'zestimate', 'rentZestimate', 'taxAssessedValue', 'DaysOnZillow',
    'city_tag', 'Sq Ft Multiplier', 'Rent Multiplier', 'Taxes','Insurance','PMT', 'PITI'  # New columns
]

# Function to calculate bounding box
def get_bounding_box(center_lat, center_long, target_area):
    half_side_length = (target_area ** 0.5) / 2
    north_point = geodesic(miles=half_side_length).destination(Point(center_lat, center_long), 0)
    south_point = geodesic(miles=half_side_length).destination(Point(center_lat, center_long), 180)
    east_point = geodesic(miles=half_side_length).destination(Point(center_lat, center_long), 90)
    west_point = geodesic(miles=half_side_length).destination(Point(center_lat, center_long), 270)
    return north_point.latitude, east_point.longitude, south_point.latitude, west_point.longitude

# Loop through each city
for city in cities:
    city_name = city["name"]
    center_lat = city["lat"]
    center_long = city["long"]
    target_area = city["target_area"]
    ne_lat, ne_long, sw_lat, sw_long = get_bounding_box(center_lat, center_long, target_area)
    page = 1

    while len(combined_results) < target_listings:
        data = pyzill.search_multi_family(page, ne_lat, ne_long, sw_lat, sw_long, zoom_value)
        page_listings = data.get("listResults", [])
        initial_count = len(combined_results)

        for listing in page_listings:
            zpid = listing.get("zpid")
            if zpid not in unique_ids:
                unique_ids.add(zpid)
                listing["city_tag"] = city_name
                combined_results.append(listing)
        
        print(f"City: {city_name}, Page {page} processed. Total unique listings so far: {len(combined_results)}")

        if len(combined_results) == initial_count:
            print(f"No new unique listings found on page {page} for {city_name}. Stopping loop.")
            break
        # END NEW

        page += 1
        if not page_listings:
            break

# Write to Excel file
wb = Workbook()
ws = wb.active
ws.append(fields)

for listing in combined_results:
    hdp_data = listing.get("hdpData", {}).get("homeInfo", {})
    
    unformatted_price = listing.get('unformattedPrice') or 0
    area = max(listing.get('area') or 0, 1)
    beds = max(listing.get('beds') or 0, 1)
    DaysOnZillow = hdp_data.get('timeOnZillow')/ (1000*60*60*24)

    sq_ft_multiplier = unformatted_price / area if area > 0 else None
    rent_multiplier = unformatted_price / (beds * Per_bed_rent) if beds > 0 else None
    
    loan_amount = unformatted_price * (1 - Down_pmt_per)

    address_city = (listing.get('addressCity') or "").strip().lower()
    county_name = city_to_county.get(address_city, "Salt Lake County")  # Default if missing
    county_tax_rate = county_tax_rates.get(county_name, 0.0055)  # Fallback default
    taxes = county_tax_rate * unformatted_price /12 #monthly amount
    insurance = insurance_rate * unformatted_price /12 #monthly amount

    monthly_pmt = PMT(loan_amount, interest_rate)
    monthly_piti = PITI(loan_amount, county_tax_rate, insurance_rate, unformatted_price)



    row = [
        listing.get('zpid'),
        listing.get('rawHomeStatusCd'),
        hdp_data.get('homeStatusForHDP'),
        hdp_data.get('homeStatus'),
        hdp_data.get('homeType'),
        listing.get('address'),
        listing.get('addressZipcode'),
        listing.get('addressCity'),
        listing.get('addressState'),
        hdp_data.get('latitude', listing.get('latLong', {}).get('latitude')),
        hdp_data.get('longitude', listing.get('latLong', {}).get('longitude')),
        listing.get('detailUrl'),
        unformatted_price,
        listing.get('beds'),
        listing.get('baths'),
        area,
        listing.get('zestimate'),
        hdp_data.get('rentZestimate'),
        hdp_data.get('taxAssessedValue'),
        DaysOnZillow,
        listing.get("city_tag"),
        sq_ft_multiplier,
        rent_multiplier,
        taxes,
        insurance,
        monthly_pmt,
        monthly_piti
    ]

    ws.append(row)



#-------- EXCEL FORMATTING -------- 
    from openpyxl.utils import get_column_letter
    # Auto-adjust column widths
    for i, col in enumerate(fields, 1):
        ws.column_dimensions[get_column_letter(i)].width = 16
    # Freeze header row
    ws.freeze_panes = "A2"
    from openpyxl.styles import numbers

    # Apply number format to relevant numeric columns
    numeric_cols = {
        "unformattedPrice", "beds", "baths", "area", "zestimate", "rentZestimate",
        "taxAssessedValue", "Sq Ft Multiplier", "DaysOnZillow","Rent Multiplier",
        "Taxes","Insurance", "PMT", "PITI"
    }

    for col_index, header in enumerate(fields, 1):
        if header in numeric_cols:
            for row_index in range(2, ws.max_row + 1):  # Skip header
                cell = ws.cell(row=row_index, column=col_index)
                cell.number_format = '#,##0'  # Comma-separated, 0 decimals
#-------- --------  -------- --------


wb.save(file_name)

# --------  Summary of Listings per City -------- 
from collections import Counter

city_counts = Counter(l['city_tag'] for l in combined_results)
print("Listings per city:")
for city, count in city_counts.items():
    print(f"  {city}: {count}")
print(f"Unique listings from all cities saved to '{file_name}'")


City: Millcreek, Page 1 processed. Total unique listings so far: 11
City: Millcreek, Page 2 processed. Total unique listings so far: 11
No new unique listings found on page 2 for Millcreek. Stopping loop.
City: Avenues, Page 1 processed. Total unique listings so far: 30
City: Avenues, Page 2 processed. Total unique listings so far: 30
No new unique listings found on page 2 for Avenues. Stopping loop.
City: Salt Lake City, Page 1 processed. Total unique listings so far: 62
City: Salt Lake City, Page 2 processed. Total unique listings so far: 90
City: Salt Lake City, Page 3 processed. Total unique listings so far: 108
City: Salt Lake City, Page 4 processed. Total unique listings so far: 108
No new unique listings found on page 4 for Salt Lake City. Stopping loop.
City: Provo, Page 1 processed. Total unique listings so far: 131
City: Provo, Page 2 processed. Total unique listings so far: 131
No new unique listings found on page 2 for Provo. Stopping loop.
Listings per city:
  Millcreek: 1

In [ ]:
X = combined_results
def get_price(listing):
    return listing.get("unformattedPrice") or 0
price = get_price(X[5])
loan_amt = price * 0.8

payment = format(PMT(loan_amt,.067),",.0f")
piti = format(PITI(loan_amt, 0.004, 0.004, price),",.0f")
format(price,",.0f"), payment, piti

332.8333333333333
332.8333333333333


('998,500', '5,154', '5,820')